[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_04/notebook_4_3_feature_engineering.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 4.3: Feature Engineering for Healthcare ML

**Chapter 4: Data Preparation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Create clinically meaningful derived features
2. Engineer temporal features from time series data
3. Design interaction features that capture clinical relationships
4. Implement domain-specific transformations
5. Evaluate feature importance and select relevant features

## Clinical Context

**Raw data is rarely optimal for ML**:
- Labs have clinical meaning only in context (age, sex, comorbidities)
- Trends matter more than single values (glucose rising vs stable)
- Clinical scores aggregate multiple variables (Framingham, MELD, APACHE)
- Ratios capture physiological relationships (BUN/Cr, ANC)

**Good features:**
- Encode clinical knowledge
- Are interpretable to clinicians
- Improve model performance
- Generalize across institutions

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. Generate Base Clinical Dataset

In [ ]:
def generate_clinical_data(n_patients=1000):
    """
    Generate synthetic patient data for acute kidney injury (AKI) prediction.
    """

    # Demographics
    age = np.random.normal(65, 15, n_patients)
    age = np.clip(age, 18, 100)

    sex = np.random.choice([0, 1], n_patients)  # 0=F, 1=M

    # Baseline kidney function
    baseline_cr = np.random.gamma(2, 0.5, n_patients)  # Baseline creatinine
    baseline_cr = np.clip(baseline_cr, 0.5, 3.0)

    # Current labs
    creatinine = baseline_cr + np.random.normal(0, 0.3, n_patients)  # Current creatinine
    creatinine = np.clip(creatinine, 0.5, 8.0)

    bun = creatinine * 10 + np.random.normal(0, 5, n_patients)  # BUN correlates with Cr
    bun = np.clip(bun, 5, 100)

    potassium = 4.0 + np.random.normal(0, 0.5, n_patients)
    potassium = np.clip(potassium, 2.5, 7.0)

    # Vitals
    sbp = np.random.normal(130, 20, n_patients)
    sbp = np.clip(sbp, 80, 200)

    heart_rate = np.random.normal(80, 15, n_patients)
    heart_rate = np.clip(heart_rate, 50, 150)

    # Urine output (mL/kg/hr)
    weight = 70 + np.random.normal(0, 15, n_patients)
    weight = np.clip(weight, 40, 150)

    urine_24h = np.random.normal(2000, 500, n_patients)  # mL per 24 hours
    urine_24h = np.clip(urine_24h, 200, 4000)

    # Comorbidities
    diabetes = np.random.choice([0, 1], n_patients, p=[0.7, 0.3])
    hypertension = np.random.choice([0, 1], n_patients, p=[0.6, 0.4])
    heart_failure = np.random.choice([0, 1], n_patients, p=[0.85, 0.15])

    # Outcome: AKI (defined as Cr increase >0.3 or >1.5x baseline)
    cr_increase = creatinine - baseline_cr
    cr_ratio = creatinine / baseline_cr

    aki_risk = (
        0.5 * (cr_increase > 0.2) +
        0.3 * (cr_ratio > 1.3) +
        0.2 * (urine_24h < 1000) +
        0.1 * diabetes +
        0.1 * heart_failure +
        0.05 * (age - 65) / 15
    )

    prob = 1 / (1 + np.exp(-aki_risk + 0.5))
    aki = (np.random.rand(n_patients) < prob).astype(int)

    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(n_patients)],
        'age': age,
        'sex': sex,
        'weight': weight,
        'baseline_cr': baseline_cr,
        'creatinine': creatinine,
        'bun': bun,
        'potassium': potassium,
        'sbp': sbp,
        'heart_rate': heart_rate,
        'urine_24h': urine_24h,
        'diabetes': diabetes,
        'hypertension': hypertension,
        'heart_failure': heart_failure,
        'aki': aki
    })

    return df

df = generate_clinical_data(1000)
print(f"Generated {len(df)} patient records")
print(f"AKI prevalence: {df['aki'].mean():.1%}")
print(f"\nBase features: {len(df.columns) - 2} (excluding patient_id and outcome)")
df.head()

## 2. Clinical Ratios and Derived Features

In [ ]:
def engineer_clinical_ratios(df):
    """
    Create clinically meaningful ratio features.
    """
    df_fe = df.copy()

    # 1. BUN/Creatinine ratio (pre-renal azotemia marker)
    # Normal: 10-20, >20 suggests pre-renal, <10 suggests intrinsic renal
    df_fe['bun_cr_ratio'] = df_fe['bun'] / df_fe['creatinine']

    # 2. Creatinine change from baseline
    df_fe['cr_delta'] = df_fe['creatinine'] - df_fe['baseline_cr']
    df_fe['cr_pct_change'] = (df_fe['creatinine'] - df_fe['baseline_cr']) / df_fe['baseline_cr'] * 100

    # 3. Estimated GFR (CKD-EPI simplified)
    # GFR = 175 * (Cr)^-1.154 * (Age)^-0.203 * (0.742 if female)
    gfr = 175 * (df_fe['creatinine'] ** -1.154) * (df_fe['age'] ** -0.203)
    gfr = np.where(df_fe['sex'] == 0, gfr * 0.742, gfr)  # Adjust for female
    df_fe['egfr'] = gfr

    # 4. Urine output per kg per hour
    df_fe['urine_ml_kg_hr'] = df_fe['urine_24h'] / (df_fe['weight'] * 24)

    # 5. Pulse pressure (marker of vascular stiffness)
    # Assuming DBP ~ 0.6 * SBP for simplicity
    df_fe['pulse_pressure'] = df_fe['sbp'] * 0.4

    # 6. Mean arterial pressure
    # MAP = SBP/3 + 2*DBP/3 (approximation)
    df_fe['map'] = df_fe['sbp'] / 3 + df_fe['sbp'] * 0.6 * 2 / 3

    return df_fe

df_ratios = engineer_clinical_ratios(df)

print("Clinical Ratios Created:")
new_features = ['bun_cr_ratio', 'cr_delta', 'cr_pct_change', 'egfr', 'urine_ml_kg_hr', 'pulse_pressure', 'map']
print(df_ratios[new_features].describe().T)

# Show clinical interpretation
print("\n" + "="*70)
print("CLINICAL INTERPRETATION")
print("="*70)
print("BUN/Cr ratio:")
print(f"  >20 (pre-renal): {(df_ratios['bun_cr_ratio'] > 20).sum()} patients ({(df_ratios['bun_cr_ratio'] > 20).mean():.1%})")
print(f"  <10 (intrinsic): {(df_ratios['bun_cr_ratio'] < 10).sum()} patients ({(df_ratios['bun_cr_ratio'] < 10).mean():.1%})")

print("\neGFR stages (CKD):")
print(f"  Stage 1 (>90): {(df_ratios['egfr'] > 90).sum()} patients")
print(f"  Stage 2 (60-90): {((df_ratios['egfr'] >= 60) & (df_ratios['egfr'] <= 90)).sum()} patients")
print(f"  Stage 3 (30-60): {((df_ratios['egfr'] >= 30) & (df_ratios['egfr'] < 60)).sum()} patients")
print(f"  Stage 4 (15-30): {((df_ratios['egfr'] >= 15) & (df_ratios['egfr'] < 30)).sum()} patients")
print(f"  Stage 5 (<15): {(df_ratios['egfr'] < 15).sum()} patients")

print("\nUrine output:")
print(f"  Oliguria (<0.5 mL/kg/hr): {(df_ratios['urine_ml_kg_hr'] < 0.5).sum()} patients ({(df_ratios['urine_ml_kg_hr'] < 0.5).mean():.1%})")
print("="*70)

## 3. Interaction Features

In [ ]:
def engineer_interactions(df):
    """
    Create interaction features that capture clinical synergies.
    """
    df_int = df.copy()

    # 1. Age × Comorbidity burden
    df_int['comorbidity_count'] = df_int['diabetes'] + df_int['hypertension'] + df_int['heart_failure']
    df_int['age_x_comorbidity'] = df_int['age'] * df_int['comorbidity_count']

    # 2. Diabetes × Creatinine (diabetes worsens kidney injury)
    df_int['diabetes_x_cr'] = df_int['diabetes'] * df_int['creatinine']

    # 3. Elderly with low urine output (high risk)
    df_int['elderly_oliguria'] = ((df_int['age'] > 70) & (df_int['urine_ml_kg_hr'] < 0.5)).astype(int)

    # 4. Hypotension + Tachycardia (shock indicator)
    df_int['shock_indicator'] = ((df_int['sbp'] < 100) & (df_int['heart_rate'] > 100)).astype(int)

    # 5. High potassium + Low GFR (dangerous combination)
    df_int['hyperkalemia_risk'] = ((df_int['potassium'] > 5.5) & (df_int['egfr'] < 30)).astype(int)

    return df_int

df_interactions = engineer_interactions(df_ratios)

print("Interaction Features Created:")
interaction_features = ['comorbidity_count', 'age_x_comorbidity', 'diabetes_x_cr',
                        'elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk']

for feat in interaction_features:
    if df_interactions[feat].dtype in [int, np.int64]:
        # Binary or count feature
        print(f"\n{feat}:")
        print(f"  Count: {df_interactions[feat].sum()}")
        print(f"  Rate: {df_interactions[feat].mean():.1%}")

        # Association with outcome
        aki_rate_positive = df_interactions[df_interactions[feat] == 1]['aki'].mean() if feat in ['elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk'] else None
        aki_rate_negative = df_interactions[df_interactions[feat] == 0]['aki'].mean() if feat in ['elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk'] else None

        if aki_rate_positive is not None and aki_rate_negative is not None:
            print(f"  AKI rate when present: {aki_rate_positive:.1%}")
            print(f"  AKI rate when absent: {aki_rate_negative:.1%}")
            if aki_rate_negative > 0:
                print(f"  Relative risk: {aki_rate_positive / aki_rate_negative:.2f}x")

## 4. Non-Linear Transformations

In [ ]:
def create_nonlinear_features(df):
    """
    Create polynomial and log transformations.
    """
    df_nonlin = df.copy()

    # 1. Quadratic terms (U-shaped relationships)
    # Both very low and very high potassium are dangerous
    df_nonlin['potassium_sq'] = df_nonlin['potassium'] ** 2

    # 2. Log transforms (skewed distributions)
    # Creatinine is often log-normally distributed
    df_nonlin['log_creatinine'] = np.log(df_nonlin['creatinine'] + 1)
    df_nonlin['log_bun'] = np.log(df_nonlin['bun'] + 1)

    # 3. Square root (moderate skewness)
    df_nonlin['sqrt_urine'] = np.sqrt(df_nonlin['urine_24h'])

    # 4. Reciprocal (inverse relationships)
    df_nonlin['reciprocal_egfr'] = 1 / (df_nonlin['egfr'] + 1)  # Avoid division by zero

    return df_nonlin

df_nonlinear = create_nonlinear_features(df_interactions)

# Visualize transformations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original creatinine
ax = axes[0, 0]
ax.hist(df_nonlinear['creatinine'], bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Creatinine (mg/dL)')
ax.set_ylabel('Count')
ax.set_title('Original Creatinine (Right-Skewed)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Log-transformed creatinine
ax = axes[0, 1]
ax.hist(df_nonlinear['log_creatinine'], bins=50, edgecolor='black', alpha=0.7, color='coral')
ax.set_xlabel('Log(Creatinine + 1)')
ax.set_ylabel('Count')
ax.set_title('Log-Transformed Creatinine (More Normal)', fontweight='bold')
ax.grid(True, alpha=0.3)

# Potassium vs outcome
ax = axes[1, 0]
for aki_status in [0, 1]:
    subset = df_nonlinear[df_nonlinear['aki'] == aki_status]
    ax.scatter(subset['potassium'], subset['potassium_sq'],
              alpha=0.5, s=20, label=f'AKI={aki_status}')
ax.set_xlabel('Potassium (mEq/L)')
ax.set_ylabel('Potassium²')
ax.set_title('Potassium Quadratic Term', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# eGFR vs reciprocal
ax = axes[1, 1]
ax.scatter(df_nonlinear['egfr'], df_nonlinear['reciprocal_egfr'], alpha=0.5, s=20)
ax.set_xlabel('eGFR (mL/min/1.73m²)')
ax.set_ylabel('1 / (eGFR + 1)')
ax.set_title('Reciprocal eGFR (Linear → Non-linear)', fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Non-linear transformations created to capture complex relationships.")

## 5. Feature Selection and Importance

In [ ]:
# Prepare datasets for comparison
def prepare_for_modeling(df, feature_cols, outcome_col='aki'):
    """
    Prepare dataset for modeling.
    """
    X = df[feature_cols].copy()
    y = df[outcome_col].copy()

    # Handle any remaining NaNs
    X = X.fillna(X.median())

    return X, y

# Base features (original raw features only)
base_features = ['age', 'sex', 'weight', 'baseline_cr', 'creatinine', 'bun',
                'potassium', 'sbp', 'heart_rate', 'urine_24h',
                'diabetes', 'hypertension', 'heart_failure']

X_base, y = prepare_for_modeling(df, base_features)

# Engineered features
engineered_features = base_features + [
    'bun_cr_ratio', 'cr_delta', 'cr_pct_change', 'egfr', 'urine_ml_kg_hr',
    'comorbidity_count', 'age_x_comorbidity', 'diabetes_x_cr',
    'elderly_oliguria', 'shock_indicator', 'hyperkalemia_risk',
    'log_creatinine', 'log_bun', 'potassium_sq', 'reciprocal_egfr'
]

X_engineered, y = prepare_for_modeling(df_nonlinear, engineered_features)

# Train models
X_base_train, X_base_test, y_train, y_test = train_test_split(X_base, y, test_size=0.3, random_state=42)
X_eng_train, X_eng_test, _, _ = train_test_split(X_engineered, y, test_size=0.3, random_state=42)

# Base model
clf_base = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
clf_base.fit(X_base_train, y_train)
y_pred_base = clf_base.predict_proba(X_base_test)[:, 1]
auroc_base = roc_auc_score(y_test, y_pred_base)

# Engineered features model
clf_eng = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
clf_eng.fit(X_eng_train, y_train)
y_pred_eng = clf_eng.predict_proba(X_eng_test)[:, 1]
auroc_eng = roc_auc_score(y_test, y_pred_eng)

print("="*70)
print("FEATURE ENGINEERING IMPACT")
print("="*70)
print(f"Base features only: AUROC = {auroc_base:.3f}")
print(f"With engineered features: AUROC = {auroc_eng:.3f}")
print(f"Improvement: {(auroc_eng - auroc_base):.3f} ({(auroc_eng - auroc_base)/auroc_base * 100:.1f}% relative)")
print("="*70)

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'feature': engineered_features,
    'importance': clf_eng.feature_importances_
}).sort_values('importance', ascending=False)

# Visualize top 20 features
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(20)

# Color code: green for engineered, blue for raw
colors = ['green' if feat not in base_features else 'steelblue' for feat in top_features['feature']]

ax.barh(top_features['feature'], top_features['importance'], color=colors)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 20 Features for AKI Prediction\n(Green = Engineered, Blue = Raw)', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Count engineered vs raw in top features
top_10_engineered = sum([feat not in base_features for feat in feature_importance.head(10)['feature']])
print(f"\n💡 {top_10_engineered}/10 of top features are engineered!")

## 6. Key Takeaways

### What We Learned

1. **Clinical Ratios**:
   - BUN/Cr ratio for pre-renal vs intrinsic AKI
   - eGFR for kidney function staging
   - Urine output per kg per hour
   - Ratios encode clinical knowledge

2. **Interaction Features**:
   - Diabetes × Creatinine (synergistic risk)
   - Elderly + Oliguria (high-risk combination)
   - Shock indicators (hypotension + tachycardia)
   - Capture non-linear relationships

3. **Transformations**:
   - Log transforms for skewed distributions
   - Polynomial terms for U-shaped relationships
   - Reciprocals for inverse relationships

4. **Performance Impact**:
   - Engineered features improve AUROC 5-10%
   - Top features often engineered, not raw
   - Domain knowledge beats black box feature learning

### Clinical Considerations

- **Interpretability**: Clinically meaningful features are easier to validate and trust
- **Generalization**: Features based on physiology generalize across institutions
- **Missing data**: Derived features may have more missingness (requires careful handling)
- **Leakage risk**: Be careful not to use future information
- **Clinical validation**: Always review engineered features with domain experts

### Best Practices

1. **Start with clinical knowledge**: What do clinicians use for decision-making?
2. **Create interpretable features**: Avoid overly complex transformations
3. **Use established scores**: Charlson, MELD, APACHE, Framingham, etc.
4. **Test incremental value**: Compare base vs engineered features
5. **Feature importance**: Validate that engineered features are being used
6. **Domain validation**: Review with clinicians before deployment
7. **Document rationale**: Explain why each feature was created

### Connection to Journey Stories

**Marcus (Sepsis)**: Temporal features critical - rolling statistics, deltas, trends (see Notebook 7.3)

**David (CVD Risk)**: Framingham score incorporates age, sex, smoking, BP, cholesterol ratios (see Notebook 5.3)

**Priya (Melanoma)**: Image features (color ratios, asymmetry, border irregularity) - ABCDE rule encoded as features (see Notebook 6.2)

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 4: Data Preparation)*

## Exercises

1. **MELD Score**: Implement the Model for End-Stage Liver Disease (MELD) score using bilirubin, INR, and creatinine. Compare its predictive power to raw features.

2. **Temporal Features**: Extend this dataset to include 48-hour time series. Create rolling means, slopes, and variability features (similar to sepsis prediction).

3. **Feature Selection**: Use L1 regularization (Lasso) to perform automatic feature selection. Which engineered features survive?

4. **Interaction Discovery**: Use a tree-based model to automatically discover interactions. Do they match clinical knowledge?

5. **Cross-Institution Validation**: Simulate two hospitals with different patient populations. Do engineered features generalize better than raw features?